# Export PhysiCell Single-Sink Diffusion CSV

This notebook extracts the PhysiCell single-cell diffusion sink output and writes a plotting-ready CSV next to the raw result files.

The exported table follows the decay exporter schema: it records the tool, run metadata, spatial resolution, sampling interval, source descriptions, time in minutes, whole-box average concentration in µM, and concentration at the voxel nearest the origin in µM.

In [26]:
from pathlib import Path
import xml.etree.ElementTree as ET

import numpy as np
import pandas as pd
from pctk import multicellds

In [27]:
ROOT = Path('/home/tntiniak/Work/observatory_benchmark')
PHYSICELL_DIR = ROOT / 'PhysiCell' / 'results' / 'diffusion_single_cell_sink'
SETTINGS_FILENAME = 'PhysiCell_settings.xml'
OUTPUT_FILENAME = 'physicell_diffusion_single_sink_plot_data.csv'

RESULTS_DIR = PHYSICELL_DIR
SETTINGS_PATH = RESULTS_DIR / SETTINGS_FILENAME
OUTPUT_PATH = RESULTS_DIR / OUTPUT_FILENAME

SETTINGS_PATH

PosixPath('/home/tntiniak/Work/observatory_benchmark/PhysiCell/results/diffusion_single_cell_sink/PhysiCell_settings.xml')

In [28]:
def make_physicell_resolution_label(dx_um: float) -> str:
    return f'voxel size={dx_um:.0f} um'


def nearest_axis_index(axis_values: np.ndarray, coordinate: float) -> int:
    return int(np.abs(axis_values - coordinate).argmin())


def load_physicell_run(results_dir: Path) -> pd.DataFrame:
    settings_root = ET.parse(SETTINGS_PATH).getroot()
    dx_um = float(settings_root.findtext('.//domain/dx'))
    dt_min = float(settings_root.findtext('.//overall/dt_diffusion'))
    output_interval_min = float(settings_root.findtext('.//save/full_data/interval'))

    reader = multicellds.MultiCellDS(output_folder=str(results_dir))
    average_uM = []
    center_uM = []
    center_index = None

    for _, microenvironment in reader.microenvironment_as_matrix_iterator():
        concentration_field = microenvironment[4]

        if center_index is None:
            x_axis = np.unique(microenvironment[0])
            y_axis = np.unique(microenvironment[1])
            z_axis = np.unique(microenvironment[2])
            dims = (len(x_axis), len(y_axis), len(z_axis))
            center_index = np.ravel_multi_index(
                (
                    nearest_axis_index(x_axis, 0.0),
                    nearest_axis_index(y_axis, 0.0),
                    nearest_axis_index(z_axis, 0.0),
                ),
                dims,
            )

        average_uM.append(concentration_field.mean() / 602.2)
        center_uM.append(concentration_field[center_index] / 602.2)

    time_min = np.round(
        np.arange(len(average_uM), dtype=float) * output_interval_min,
        2,
    )
    run_id = f'PhysiCell_dx_{int(dx_um)}um_single_sink'

    return pd.DataFrame({
        'tool': 'PhysiCell',
        'run_id': run_id,
        'dx_um': dx_um,
        'dt_min': dt_min,
        'average_source': 'mean over all voxels from microenvironment[4]',
        'center_source': 'single voxel nearest the origin from microenvironment[4]',
        'resolution_label': make_physicell_resolution_label(dx_um),
        'timestep': time_min,
        'average_uM': np.array(average_uM, dtype=float),
        'center_uM': np.array(center_uM, dtype=float),
    })

In [29]:
frame = load_physicell_run(RESULTS_DIR)
frame.to_csv(OUTPUT_PATH, index=False)

pd.DataFrame({
    'csv_path': [str(OUTPUT_PATH)],
    'rows_written': [len(frame)],
    'time_start_min': [frame['timestep'].iloc[0]],
    'time_end_min': [frame['timestep'].iloc[-1]],
    'dx_um': [frame['dx_um'].iloc[0]],
    'dt_min': [frame['dt_min'].iloc[0]],
})

,csv_path,rows_written,time_start_min,time_end_min,dx_um,dt_min
0,/home/tntiniak/Work/observatory_benchmark/Phys...,1001,0.0,10.0,20.0,0.01
